# Fine-tune a small model on Kaggle

Original notebook for finetune-anywhere. Start with the JSON extractor, then replace its training data with your own licensed examples. Keep evaluation examples separate. This notebook needs an NVIDIA GPU; availability and session duration are not guaranteed.

Select a GPU runtime before running. On Kaggle also enable Internet in notebook settings. On Colab use Runtime > Change runtime type > GPU. Run cells in order. Dependency pins are resolver-checked; see MAINTAINER_NOTES.md for hardware validation status.

In [ ]:
!nvidia-smi

## Get the starter

Clone the repository so model names, datasets and training code stay consistent. Re-running this cell reuses the checkout.

In [ ]:
from pathlib import Path
import subprocess
root = Path('/content/finetune-anywhere' if 'kaggle' == 'colab' else '/kaggle/working/finetune-anywhere')
if not root.exists():
    subprocess.run(['git', 'clone', 'https://github.com/junaiddshaukat/finetune-anywhere.git', str(root)], check=True)
%cd $root

## Install the training environment

The shared requirements pin the training stack. If a package was already imported, restart the session after this cell, then rerun the checkout cell before continuing.

In [ ]:
%pip install -r paths/nvidia-gpu/requirements.txt

## Choose a persistent export location

Files under /kaggle/working are included when you save a notebook version with outputs. Save a version and download the model before ending the session. Session storage alone is not a backup.

In [ ]:
output = Path('/kaggle/working/outputs/json-extractor')

## Optional Hub authentication

The default model is public and needs no token. To upload later, add a Kaggle secret named HF_TOKEN and enable it for this notebook. Never paste credentials into a cell.

In [ ]:
upload_to_hub = False
if upload_to_hub:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(token=UserSecretsClient().get_secret('HF_TOKEN'), add_to_git_credential=False)

## Validate the data

This catches format errors and prompts shared between splits. Character-based token estimates are conservative hints; use --tokenizer starter for exact template-aware counts.

In [ ]:
!python scripts/validate_dataset.py recipes/json-extractor/data/train.jsonl recipes/json-extractor/data/valid.jsonl recipes/json-extractor/data/eval.jsonl

## Train and export

One epoch is a starting point, not proof of quality. The script saves adapters, merged weights and a quantized GGUF. Keep the held-out evaluation set out of training. For an installation smoke test add --max-steps 2; then run a complete training job in a new output folder.

In [ ]:
subprocess.run(['python', 'paths/nvidia-gpu/train_unsloth.py', '--config', 'recipes/json-extractor/config.yaml', '--output', str(output), '--export-gguf'], check=True)

## Locate the portable model

Download the GGUF to your laptop. GGUF conversion may require extra RAM, disk and a C++ build toolchain. If export runs out of memory, retain merged weights and follow guides/05-quantize.md on a larger machine.

In [ ]:
gguf_files = list(output.rglob('*.gguf'))
for file in gguf_files:
    print(file, round(file.stat().st_size / 1e9, 2), 'GB')
if not gguf_files:
    raise RuntimeError('No GGUF found. Read the export log before continuing.')

Use the Files panel to download the GGUF from the output directory, or Save Version with output files. See guides/07-share-your-model.md for a Hub upload.

## Evaluate and run locally

Follow guides/04-evaluate.md to compare identical held-out prompts on the base and fine-tuned model. Then use guides/06-run-locally.md to import the GGUF into Ollama, llama.cpp or LM Studio. Publish actual outputs and failures, not just a falling training loss.

To resume, pass --resume and the saved checkpoint directory to the training command. Keep the same model, recipe and training settings. See guides/07-share-your-model.md to upload the artifact and add a showcase entry.